In [ ]:
import os, pickle, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.axes as axes
import pprint
from tqdm import tqdm
import multiprocessing
import re
from seaborn import heatmap
from functools import reduce
import seaborn as sns
import pandas as pd

In [ ]:
def traverse_directory(root_directory):
    """Traverse directory and collect all .pkl file paths."""
    file_paths = []
    for dirpath, _, filenames in os.walk(root_directory):
        for filename in filenames:
            if filename.endswith(".pkl") and "subgraph" not in filename:  # Adjust based on your file format
                file_paths.append(os.path.join(dirpath, filename))
    return file_paths

def extract_model_and_relation(file_path):
    """Extract model name and relation name from file path."""
    path_parts = file_path.split(os.sep)
    model_name = path_parts[-2]  # Model name is the second last part of the path
    relation_name = os.path.splitext(path_parts[-1])[0]  # File name without extension
    return model_name, relation_name

def read_single_file(file_path):
    """Load data from a single .pkl file and return the model, relation, and data."""
    model_name, relation_name = extract_model_and_relation(file_path)
    with open(file_path, 'rb') as file:
        data_entries = pickle.load(file)
    return model_name, relation_name, data_entries

def read_all_files(file_paths):
    """Reads all .pkl files and organizes data by model and relation sequentially."""
    models_data = {}
    total_files = len(file_paths)
    
    print(f"Total files to process: {total_files}")

    # Sequentially read each file
    for file_path in tqdm(file_paths, total=total_files, desc="Processing files"):
        model_name, relation_name, data_entries = read_single_file(file_path)
        if model_name not in models_data:
            models_data[model_name] = {}
        models_data[model_name][relation_name] = data_entries

    return models_data

In [ ]:
root_directory = "/nfs/datz/olmo_models/processed_outputs/"

file_paths = traverse_directory(root_directory)
all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")


In [ ]:
import torch
from transformer_lens import HookedTransformer

def load_models(model_name, device='cuda:4', checkpoint=""):
    model = HookedTransformer.from_pretrained_no_processing(
        model_name,
        dtype=torch.bfloat16,
        center_unembed=True,
        center_writing_weights=True,
        #fold_ln=True,
        device=device,
        trust_remote_code=True,
        cache_dir="/mounts/data/proj/hypersum",
        checkpoint_value=checkpoint,
    )

    return model


# Load the models
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:1', checkpoint="main")

In [ ]:
models_data['main']["country_capital_city"][0]

In [ ]:
models_data['main']["country_capital_city"][0]['output_top_tokens'][55]

In [ ]:
sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )

In [ ]:
# def calculate_overlap_with_main(heatmaps_dict, THRESHOLD, fact_idx):
#     overlap_results = {}
    
#     # Retrieve the 'main' model's heatmaps
#     main_heatmaps = heatmaps_dict['main']

#     # Iterate through each model (excluding 'main') and each heatmap type
#     for model_name, model_data in heatmaps_dict.items():
#         if model_name == 'main':
#             continue

#         # Store overlaps for this model
#         model_overlaps = {}
        
#         for heatmap_type in main_heatmaps.keys():
#             # Perform element-wise overlap calculation between model's heatmap and 'main' heatmap
#             if heatmap_type == "answer_heatmap":
#                 overlap_map = (main_heatmaps[heatmap_type][fact_idx] - model_data[heatmap_type][fact_idx]) < THRESHOLD #(model_data[heatmap_type][fact_idx] > THRESHOLD) == (main_heatmaps[heatmap_type][fact_idx] > THRESHOLD)
#             else:
#                 overlap_map =  (main_heatmaps[heatmap_type] - model_data[heatmap_type]) < THRESHOLD #(model_data[heatmap_type] > THRESHOLD) == (main_heatmaps[heatmap_type] > THRESHOLD)
            
#             # Calculate summary statistics, e.g., proportion of overlapping positions
#             proportion_overlap = np.sum(overlap_map) / overlap_map.size
            
#             # Store the overlap map and proportion
#             model_overlaps[heatmap_type] = {
#                 'overlap_map': overlap_map,
#                 'consistency': proportion_overlap
#             }
        
#         # Add this model's overlap results to the main dictionary
#         overlap_results[model_name] = model_overlaps
    
#     return overlap_results

# def calculate_switches(heatmaps_dict, THRESHOLD, fact_idx):
#     switches_results = {}
    
#     # Retrieve the 'main' model's heatmaps
#     #main_heatmaps = heatmaps_dict['main']

#     # Iterate through each model (excluding 'main') and each heatmap type
    
#     for i in range(len(sorted_models) - 1):
        
#         if sorted_models[i] == 'main':
#             continue
        
#         model_name = sorted_models[i]
#         next_model_name = sorted_models[i + 1]
    
#         model_data = heatmaps_dict[model_name]
#         next_model_data = heatmaps_dict[next_model_name]
    
#     # for model_name, model_data in heatmaps_dict.items():
#     #    if model_name == 'main':
#     #        continue

#         # Store overlaps for this model
#         model_switches = {}
        
#         for heatmap_type in model_data.keys():
#             # Perform element-wise overlap calculation between model's heatmap and 'main' heatmap
            
#             if heatmap_type == "answer_heatmap":
#                 switches_map = (next_model_data[heatmap_type][fact_idx] - model_data[heatmap_type][fact_idx]) > THRESHOLD #(model_data[heatmap_type][fact_idx] > THRESHOLD) != (next_model_data[heatmap_type][fact_idx] > THRESHOLD)
#             else:
#                 switches_map = (next_model_data[heatmap_type] - model_data[heatmap_type]) > THRESHOLD #(model_data[heatmap_type] > THRESHOLD) != (next_model_data[heatmap_type] > THRESHOLD)
            
#             # Calculate summary statistics, e.g., proportion of overlapping positions
#             proportion_switches = np.sum(switches_map) / switches_map.size
            
#             # Store the overlap map and proportion
#             model_switches[heatmap_type] = {
#                 'switches_map': switches_map,
#                 'proportion_switches': proportion_switches
#             }
        
#         # Add this model's overlap results to the main dictionary
#         switches_results[model_name] = model_switches
    
#     return switches_results


# def plot_proportion_overlap_multiple(overlap_results):
#     # Prepare data
#     model_names = sorted(
#         overlap_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )
#     general_proportions = []
#     entity_proportions = []
#     answer_proportions = []
#     answer_all_proportions = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_proportions.append(overlap_results[model_name]['general_heatmap']['consistency'])
#         entity_proportions.append(overlap_results[model_name]['entity_heatmap']['consistency'])
#         answer_proportions.append(overlap_results[model_name]['answer_heatmap']['consistency'])
#         answer_all_proportions.append(overlap_results[model_name]['answer_allheatmap']['consistency'])

#     # Plotting
#     plt.figure(figsize=(14, 7))
#     plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
#     plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
#     plt.plot(model_names, answer_proportions, label="Answer Heatmap", marker='^', linestyle='-.')
#     plt.plot(model_names, answer_all_proportions, label="Answer All Heatmap", marker='x', linestyle='-.')

#     plt.xticks(rotation=90)
#     plt.xlabel("Model Steps")
#     plt.ylabel("Consistency")
#     plt.title("Consistency Across Models")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()
    
# def plot_proportion_switches(switches):
#     # Prepare data
#     model_names = sorted(
#         switches_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )
#     general_proportions = []
#     entity_proportions = []
#     answer_proportions = []
#     answer_all_proportions = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_proportions.append(switches[model_name]['general_heatmap']['proportion_switches'])
#         entity_proportions.append(switches[model_name]['entity_heatmap']['proportion_switches'])
#         answer_proportions.append(switches[model_name]['answer_heatmap']['proportion_switches'])
#         answer_all_proportions.append(switches[model_name]['answer_allheatmap']['proportion_switches'])

#     # Plotting
#     plt.figure(figsize=(14, 7))
#     plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
#     plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
#     plt.plot(model_names, answer_proportions, label="Answer Heatmap", marker='^', linestyle='-.')
#     plt.plot(model_names, answer_all_proportions, label="Answer All Heatmap", marker='x', linestyle='-.')

#     plt.xticks(rotation=90)
#     plt.xlabel("Model Steps")
#     plt.ylabel("Proportion Switches")
#     plt.title("Proportion Switches Comparison Across Models")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

# Those following cells

In [ ]:
heatmaps_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]
    
    print(model_name)    
    g_total = 0
    general_heatmap = np.zeros((32, 32), dtype=float)
    e_total = 0
    entity_heatmap = np.zeros((32, 32), dtype=float)
    
    #answer_allheatmap = np.zeros((32, 32), dtype=float)  # Changed to float for saving purposes
    #a_total = 0

    # Initialize a sub-dictionary to store answer_heatmaps per fact (by data_idx)
    relation_answer_heatmaps = {}
    relation_answer_with_specific = {}

    for idx, (relation_name, entries) in enumerate(relations_data.items()):
        
        # if relation_name != "country_capital_city":
        #     continue
        
        relation_answer_heatmap = np.zeros((32, 32), dtype=float)
        relation_answer_total = 0

        # Group entries by data_idx
        data_idx_groups = {}
        for entry in entries:
            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()
            
            data_idx = entry['data_idx']
            if data_idx not in data_idx_groups:
                data_idx_groups[data_idx] = []
            data_idx_groups[data_idx].append(entry)

        answer_specific_heatmaps = {}
        # Process each data_idx group
        for data_idx, group_entries in data_idx_groups.items():
            
            answer_specific_heatmap = np.zeros((32, 32), dtype=float)
            answer_specific_total = 0
            
            for g_entry in group_entries:
                
                general_heatmap += np.sum(entry['attnheads_contribution_maps'], axis=0)
                g_total += len(entry['attnheads_contribution_maps'])
                
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in g_entry["subj_token_span"]:
                    if t == 0:
                        e_total -= 1
                        continue
                    one_data_map += g_entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                e_total += len(g_entry["subj_token_span"])
                
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in g_entry["answer_token_span"]:
                    one_data_map += g_entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                relation_answer_heatmap += one_data_map
                answer_specific_heatmap += one_data_map
                
                e_total += len(g_entry["answer_token_span"])
                relation_answer_total += len(g_entry["answer_token_span"])
                answer_specific_total += len(g_entry["answer_token_span"])
            
            answer_specific_heatmap /= answer_specific_total
            answer_specific_heatmaps[data_idx] = answer_specific_heatmap
                
                
        relation_answer_heatmap /= relation_answer_total
        relation_answer_heatmaps[f'{relation_name}'] = relation_answer_heatmap # if its easier we can use idx just used the names for debug
        relation_answer_with_specific[f'{relation_name}'] = answer_specific_heatmaps     
    
    # Normalize heatmaps
    general_heatmap /= g_total
    entity_heatmap /= e_total

    # Store heatmaps in the dictionary, including answer_heatmaps for each fact
    heatmaps_dict[model_name] = {
        'general_heatmap': general_heatmap,
        'entity_heatmap': entity_heatmap,
        'relation_answer_heatmaps': relation_answer_heatmaps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

This is with the new version

In [ ]:
def calculate_overlap_with_main(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
                overlap_map = (main_heatmap - model_heatmap) < THRESHOLD
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
                overlap_map = (main_heatmap - model_heatmap) < THRESHOLD
            else:
                # General or entity heatmaps
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
                overlap_map = (main_heatmap - model_heatmap) < THRESHOLD

            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            
            # Store the overlap map and proportion
            model_overlaps[heatmap_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

def calculate_switches(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    switches_results = {}

    # Iterate through the sorted models (excluding 'main') in pairs
    for i in range(len(sorted_models) - 1):
        if sorted_models[i] == 'main':
            continue
        
        model_name = sorted_models[i]
        next_model_name = sorted_models[i + 1]
        
        model_data = heatmaps_dict[model_name]
        next_model_data = heatmaps_dict[next_model_name]
        
        # Store switches for this model
        model_switches = {}
        
        for heatmap_type in model_data.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                model_heatmap = model_data[heatmap_type][relation_name]
                next_model_heatmap = next_model_data[heatmap_type][relation_name]
                switches_map = (next_model_heatmap - model_heatmap) > THRESHOLD
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
                next_model_heatmap = next_model_data[heatmap_type][relation_name][fact_idx]
                switches_map = (next_model_heatmap - model_heatmap) > THRESHOLD
            else:
                # General or entity heatmaps
                model_heatmap = model_data[heatmap_type]
                next_model_heatmap = next_model_data[heatmap_type]
                switches_map = (next_model_heatmap - model_heatmap) > THRESHOLD

            # Calculate summary statistics, e.g., proportion of switching positions
            proportion_switches = np.sum(switches_map) / switches_map.size
            
            # Store the switches map and proportion
            model_switches[heatmap_type] = {
                'switches_map': switches_map,
                'proportion_switches': proportion_switches
            }
        
        # Add this model's switches results to the main dictionary
        switches_results[model_name] = model_switches
    
    return switches_results


In [ ]:
import matplotlib.pyplot as plt

def plot_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['consistency'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['consistency'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title(f"Consistency Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_proportion_switches(switches_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        switches_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(switches_results[model_name]['general_heatmap']['proportion_switches'])
        entity_proportions.append(switches_results[model_name]['entity_heatmap']['proportion_switches'])
        answer_all_proportions.append(switches_results[model_name]['relation_answer_heatmaps']['proportion_switches'])
        answer_proportions.append(switches_results[model_name]['relation_answer_with_specific']['proportion_switches'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Proportion Switches")
    plt.title(f"Proportion Switches Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
THRESHOLD = 0.2
relation_name = "country_capital_city"
fact_idx = 0
overlap = calculate_overlap_with_main(heatmaps_dict, THRESHOLD, relation_name, fact_idx)
switches = calculate_switches(heatmaps_dict, THRESHOLD, relation_name, fact_idx)


plot_proportion_overlap_multiple(overlap, relation_name, fact_idx)
plot_proportion_switches(switches, relation_name, fact_idx)

# Until here

In [ ]:
overlap['step5000-tokens20B']['relation_answer_heatmaps']

In [ ]:
overlap['step650650-tokens2728B']['relation_answer_heatmaps']

In [ ]:
model_data['general_contrib']

In [ ]:
def plot_proportion_overlap_multiple(overlap_results):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['consistency'])
        answer_proportions.append(overlap_results[model_name]['answer_heatmap']['consistency'])
        answer_all_proportions.append(overlap_results[model_name]['answer_allheatmap']['consistency'])
        
        #This was needed without the fact_idx
        # Process answer_heatmap as it is a dictionary of individual heatmaps
        # for answer_key, answer_data in overlap_results[model_name]['answer_heatmap'].items():
        #     if answer_key not in answer_proportions:
        #         answer_proportions[answer_key] = []
        #     answer_proportions[answer_key].append(answer_data['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_proportions, label="Answer Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Answer All Heatmap", marker='^', linestyle='-.')

    # Plot each individual answer heatmap consistency
    # for answer_key, proportions in answer_proportions.items():
    #     plt.plot(model_names, proportions, label=f"Answer Heatmap - {answer_key}", marker='x', linestyle='--')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title("Consistency Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## Output for country_capital (one of the best performing ones)

In [ ]:
THRESHOLD = 0.2
fact_idx = 0

overlap = calculate_overlap_with_main(heatmaps_dict, THRESHOLD, fact_idx)
switches_results = calculate_switches(heatmaps_dict, THRESHOLD, fact_idx)
plot_proportion_overlap_multiple(overlap)
plot_proportion_switches(switches_results)

## Output for person_father (bad performing one)

In [ ]:
THRESHOLD = 0.2
fact_idx = 0

overlap = calculate_overlap_with_main(heatmaps_dict, THRESHOLD, fact_idx)
switches_results = calculate_switches(heatmaps_dict, THRESHOLD, fact_idx)
plot_proportion_overlap_multiple(overlap)
plot_proportion_switches(switches_results)

## Analysis of Switches

In [ ]:
switches_results

In [ ]:
def plot_proportion_heads(model_name, heatmap_dict, fact_idx, THRESHOLD):
        
    # Define the vectors
    A = (heatmaps_dict[model_name]['general_heatmap'] > THRESHOLD).sum(axis=-1)
    B = (heatmaps_dict[model_name]['entity_heatmap'] > THRESHOLD).sum(axis=-1)
    C = (heatmap_dict['main']['answer_heatmap'][fact_idx] > THRESHOLD).sum(axis=-1)
    D = (heatmaps_dict[model_name]['answer_allheatmap'] > THRESHOLD).sum(axis=-1)

    max_value = 32

    # Compute contributions
    Contrib_A = A
    Contrib_B = B - A
    Contrib_C = C - B
    Contrib_D = D - C

    # Verify that the sum of contributions equals D
    Total_Contrib = Contrib_A + Contrib_B + Contrib_C + Contrib_D
    assert np.allclose(Total_Contrib, D), "Total contributions do not sum up to D"

    # Normalize contributions by the max value for proportional height
    Height_A = Contrib_A / max_value
    Height_B = Contrib_B / max_value
    Height_C = Contrib_C / max_value
    Height_D = Contrib_D / max_value

    # Prepare the indices for plotting
    indices = np.arange(1, 33)  # Indices from 1 to 32

    # Plot the stacked bar chart
    plt.figure(figsize=(14, 7))

    plt.bar(indices, Height_A, color='mediumvioletred', label='General Heads')
    plt.bar(indices, Height_B, bottom=Height_A, color='orange', label='Entity Heads')
    plt.bar(indices, Height_C, bottom=Height_A + Height_B, color='green', label='Answer Heads')
    plt.bar(indices, Height_D, bottom=Height_A + Height_B + Height_C, color='skyblue', label='Fact-dependent Heads')

    plt.xlabel('Layer')
    plt.ylabel('Proportion of Heads for relation')
    plt.title(f'Proportional Bar Plot of Contributions at Each Index {model_name}')
    plt.xticks(indices)
    plt.legend()

    plt.tight_layout()
    # if not os.path.exists(f'plots/{mode}/'):
    #     os.makedirs(f'plots/{mode}/')
    #plt.savefig(f'plots/{mode}/{model_name}-heads-contrib.png')
    plt.show()


In [ ]:
for x in sorted_models:
    plot_proportion_heads(x, heatmaps_dict, 0, 0.2)

In [ ]:
for x in sorted_models:
    plot_proportion_heads(x, heatmaps_dict, 0, 0.1)

In [ ]:
def calculate_consistency_pos(heatmaps, sorted_models, THRESHOLD):
    main_model = sorted_models[-1]  # The last model is the "main" model
    main_heatmap = heatmaps[main_model]['general_heatmap']

    consistency_results = {}

    for model in sorted_models[:-1]:  # Exclude the "main" model from comparison
        current_heatmap = heatmaps[model]['general_heatmap']
        consistency = main_heatmap - current_map < THRESHOLD #(current_heatmap == main_heatmap)
        consistency_results[model] = consistency

    return consistency_results

In [ ]:
consistency_results = calculate_consistency_pos(heatmaps_dict, sorted_models, 0.2)
consistency_results

In [ ]:
def plot_filtered_consistency(overall_general_heatmap, consistency_results, sorted_models):
    # Convert overall_general_heatmap to boolean type
    overall_general_heatmap_bool = overall_general_heatmap.astype(bool)

    # Get the indices of interesting layer x head combinations
    interesting_indices = np.argwhere(overall_general_heatmap_bool)
    layers, heads = overall_general_heatmap_bool.shape

    # Prepare a dataframe to hold the transformed data
    data = []

    for model in sorted_models[:-1]:  # Exclude the main model
        for layer, head in interesting_indices:
            value = consistency_results[model][layer, head]
            data.append({'Model': model, 'Layer-Head': f'L{layer}-H{head}', 'Consistency': int(value)})

    # Convert to a dataframe
    df = pd.DataFrame(data)

    df['Layer-Head'] = pd.Categorical(df['Layer-Head'], categories=sorted(df['Layer-Head'].unique(), key=lambda x: (int(x.split('-')[0][1:]), int(x.split('-')[1][1:]))), ordered=True)
    df['Model'] = pd.Categorical(df['Model'], categories=sorted_models[:-1], ordered=True)

    # Pivot the dataframe for heatmap plotting
    heatmap_data = df.pivot(index='Layer-Head', columns='Model', values='Consistency')


    # Plot the heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        heatmap_data,
        cmap='viridis',
        cbar_kws={'label': 'Consistency'},
        linewidths=0.5,
        linecolor='gray'
    )
    plt.title('Filtered Consistency Across Models', fontsize=16)
    plt.xlabel('Model')
    plt.ylabel('Layer-Head')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_filtered_consistency(overall_general_heatmap, consistency_results, sorted_models)

In [ ]:
def plot_all_layers_and_heads(consistency_results, sorted_models):
    # Get the dimensions of the general heatmap
    layers, heads = next(iter(consistency_results.values())).shape

    # Prepare a dataframe to hold the transformed data
    data = []

    for model in sorted_models[:-1]:  # Exclude the main model
        for layer in range(layers):
            for head in range(heads):
                value = consistency_results[model][layer, head]
                data.append({'Model': model, 'Layer-Head': f'L{layer}-H{head}', 'Consistency': int(value)})

    # Convert to a dataframe
    df = pd.DataFrame(data)

    # Preserve the order of Layer-Head and Model
    df['Layer-Head'] = pd.Categorical(df['Layer-Head'], categories=sorted(df['Layer-Head'].unique(), key=lambda x: (int(x.split('-')[0][1:]), int(x.split('-')[1][1:]))), ordered=True)
    df['Model'] = pd.Categorical(df['Model'], categories=sorted_models[:-1], ordered=True)

    # Pivot the dataframe for heatmap plotting
    heatmap_data = df.pivot(index='Layer-Head', columns='Model', values='Consistency')

    # Plot the heatmap
    plt.figure(figsize=(len(sorted_models) * 2, len(heatmap_data.index) // 8))
    sns.heatmap(
        heatmap_data,
        cmap='viridis',
        cbar_kws={'label': 'Consistency'},
        linewidths=0.5,
        linecolor='gray'
    )

    # Ensure all labels are shown
    plt.title('Consistency Across All Layers and Heads', fontsize=16)
    plt.xlabel('Model')
    plt.ylabel('Layer-Head')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()

# Example usage:
# 

In [ ]:
plot_all_layers_and_heads(consistency_results, sorted_models)

In [ ]:
def calculate_switches(consistency_results, sorted_models):
    # Get the dimensions of the general heatmap
    layers, heads = next(iter(consistency_results.values())).shape

    # Prepare a dataframe to hold the transformed data
    data = []

    # Iterate over consecutive model pairs
    for i in range(len(sorted_models) - 2):  # Exclude the main model
        model_a = sorted_models[i]
        model_b = sorted_models[i + 1]

        for layer in range(layers):
            for head in range(heads):
                switch = int(consistency_results[model_a][layer, head] != consistency_results[model_b][layer, head])
                data.append({
                    'Model Pair': f'{model_a} → {model_b}',
                    'Layer-Head': f'L{layer}-H{head}',
                    'Switch': switch
                })

    # Convert to a dataframe
    df = pd.DataFrame(data)

    # Preserve the order of Layer-Head and Model Pair
    df['Layer-Head'] = pd.Categorical(df['Layer-Head'], categories=sorted(df['Layer-Head'].unique(), key=lambda x: (int(x.split('-')[0][1:]), int(x.split('-')[1][1:]))), ordered=True)
    df['Model Pair'] = pd.Categorical(df['Model Pair'], categories=[f'{sorted_models[i]} → {sorted_models[i+1]}' for i in range(len(sorted_models) - 2)], ordered=True)

    return df

def plot_switches(df):
    # Pivot the dataframe for heatmap plotting
    heatmap_data = df.pivot(index='Layer-Head', columns='Model Pair', values='Switch')

    # Plot the heatmap
    plt.figure(figsize=(len(heatmap_data.columns) * 2, len(heatmap_data.index) // 8))
    sns.heatmap(
        heatmap_data,
        cmap='viridis',
        cbar_kws={'label': 'Switch'},
        linewidths=0.5,
        linecolor='gray'
    )

    # Ensure all labels are shown
    plt.title('Switches Across Models for All Layers and Heads', fontsize=16)
    plt.xlabel('Model Pair')
    plt.ylabel('Layer-Head')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
switches_df = calculate_switches(consistency_results, sorted_models)
plot_switches(switches_df)

In [ ]:
THRESHOLD = 0.2
# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_general_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        general_heatmap = heatmaps_dict[model_name]['general_heatmap'] > THRESHOLD
        overall_general_heatmap *= general_heatmap  # Element-wise multiplication



# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_entity_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        entity_heatmap = heatmaps_dict[model_name]['entity_heatmap'] > THRESHOLD
        overall_entity_heatmap *= entity_heatmap  # Element-wise multiplication
        
        
# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_answer_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        answer_heatmap = heatmaps_dict[model_name]['answer_heatmap'][0] > THRESHOLD
        overall_answer_heatmap *= answer_heatmap  # Element-wise multiplication
        

# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_answer_allheatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        answer_allheatmap = heatmaps_dict[model_name]['answer_allheatmap'] > THRESHOLD
        overall_answer_allheatmap *= answer_allheatmap  # Element-wise multiplication

In [ ]:
# Plotting the heatmaps
plt.figure(figsize=(16, 10))

# Overall General Heatmap
plt.subplot(2, 2, 1)
plt.title("Overall General Heatmap")
plt.imshow(overall_general_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Entity Heatmap
plt.subplot(2, 2, 2)
plt.title("Overall Entity Heatmap")
plt.imshow(overall_entity_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Answer Heatmap
plt.subplot(2, 2, 3)
plt.title("Overall Answer Heatmap")
plt.imshow(overall_answer_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Answer All Heatmap
plt.subplot(2, 2, 4)
plt.title("Overall Answer All Heatmap")
plt.imshow(overall_answer_allheatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_general_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        general_heatmap = heatmaps_dict[model_name]['general_heatmap'] > THRESHOLD
        overall_general_heatmap *= general_heatmap  # Element-wise multiplication



# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_entity_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        entity_heatmap = heatmaps_dict[model_name]['entity_heatmap'] > THRESHOLD
        overall_entity_heatmap *= entity_heatmap  # Element-wise multiplication
        
        
# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_answer_heatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        answer_heatmap = heatmaps_dict[model_name]['answer_heatmap'][0] > THRESHOLD
        overall_answer_heatmap *= answer_heatmap  # Element-wise multiplication
        

# Calculate the overall general heatmap by multiplying all individual general heatmaps
overall_answer_allheatmap = np.ones((32, 32), dtype=float)  # Start with ones for multiplication

for model_name in sorted_models:
    if model_name in heatmaps_dict:  # Ensure the model exists in the dictionary
        answer_allheatmap = heatmaps_dict[model_name]['answer_allheatmap'] > THRESHOLD
        overall_answer_allheatmap *= answer_allheatmap  # Element-wise multiplication

### Person father

In [ ]:
# Plotting the heatmaps
plt.figure(figsize=(16, 10))

# Overall General Heatmap
plt.subplot(2, 2, 1)
plt.title("Overall General Heatmap")
plt.imshow(overall_general_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Entity Heatmap
plt.subplot(2, 2, 2)
plt.title("Overall Entity Heatmap")
plt.imshow(overall_entity_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Answer Heatmap
plt.subplot(2, 2, 3)
plt.title("Overall Answer Heatmap")
plt.imshow(overall_answer_heatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

# Overall Answer All Heatmap
plt.subplot(2, 2, 4)
plt.title("Overall Answer All Heatmap")
plt.imshow(overall_answer_allheatmap, cmap='hot', interpolation='nearest')
plt.colorbar()

plt.tight_layout()
plt.show()

### FFN Contribution Analysis

In [ ]:
def calculate_overlap_with_main_ffn(ffns_dict, THRESHOLD, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's contributions
    main_contribs = ffns_dict['main']

    # Iterate through each model (excluding 'main') and each contribution type
    for model_name, model_data in ffns_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for contrib_type in main_contribs.keys():
            # Perform element-wise overlap calculation between model's contrib and 'main' contrib
            if contrib_type == "answer_contrib":
                overlap_map = (model_data[contrib_type][fact_idx] > THRESHOLD) == (main_contribs[contrib_type][fact_idx] > THRESHOLD)
            else:
                overlap_map = (model_data[contrib_type] > THRESHOLD) == (main_contribs[contrib_type] > THRESHOLD)
            
            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            
            # Store the overlap map and proportion
            model_overlaps[contrib_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

def calculate_switches_ffn(ffns_dict, THRESHOLD, fact_idx):
    switches_results = {}
    
    # Iterate through each model (excluding 'main') and each contribution type
    for i in range(len(sorted_models) - 1):
        if sorted_models[i] == 'main':
            continue
        
        model_name = sorted_models[i]
        next_model_name = sorted_models[i + 1]
    
        model_data = ffns_dict[model_name]
        next_model_data = ffns_dict[next_model_name]

        # Store switches for this model
        model_switches = {}
        
        for contrib_type in model_data.keys():
            # Perform element-wise switch calculation between consecutive models' contrib
            if contrib_type == "answer_contrib":
                switches_map = model_data[contrib_type][fact_idx] - next_model_data[contrib_type][fact_idx] > THRESHOLD #(model_data[contrib_type][fact_idx] > THRESHOLD) != (next_model_data[contrib_type][fact_idx] > THRESHOLD)
            else:
                switches_map = model_data[contrib_type] - next_model_data[contrib_type] > THRESHOLD #(model_data[contrib_type] > THRESHOLD) != (next_model_data[contrib_type] > THRESHOLD)
            
            # Calculate summary statistics, e.g., proportion of switching positions
            proportion_switches = np.sum(switches_map) / switches_map.size
            
            # Store the switches map and proportion
            model_switches[contrib_type] = {
                'switches_map': switches_map,
                'proportion_switches': proportion_switches
            }
        
        # Add this model's switches results to the main dictionary
        switches_results[model_name] = model_switches
    
    return switches_results


def plot_proportion_overlap_multiple_ffn(overlap_results):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_proportions = []
    answer_all_proportions = []

    # Populate lists for each contrib type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_contrib']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_contrib']['consistency'])
        answer_proportions.append(overlap_results[model_name]['answer_contrib']['consistency'])
        answer_all_proportions.append(overlap_results[model_name]['answer_allcontrib']['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Contribution", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Contribution", marker='s', linestyle='--')
    plt.plot(model_names, answer_proportions, label="Answer Contribution", marker='^', linestyle='-.')
    plt.plot(model_names, answer_all_proportions, label="Answer All Contribution", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title("Consistency Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_proportion_switches_ffn(switches_results):
    # Prepare data
    model_names = sorted(
        switches_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_proportions = []
    answer_all_proportions = []

    # Populate lists for each contrib type
    for model_name in model_names:
        general_proportions.append(switches_results[model_name]['general_contrib']['proportion_switches'])
        entity_proportions.append(switches_results[model_name]['entity_contrib']['proportion_switches'])
        answer_proportions.append(switches_results[model_name]['answer_contrib']['proportion_switches'])
        answer_all_proportions.append(switches_results[model_name]['answer_allcontrib']['proportion_switches'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Contribution", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Contribution", marker='s', linestyle='--')
    plt.plot(model_names, answer_proportions, label="Answer Contribution", marker='^', linestyle='-.')
    plt.plot(model_names, answer_all_proportions, label="Answer All Contribution", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Proportion Switches")
    plt.title("Proportion Switches Comparison Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
ffns_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]
    
    print(model_name)    
    g_total = 0
    general_contrib = np.zeros(32, dtype=float)
    e_total = 0
    entity_contrib = np.zeros(32, dtype=float)
    answer_allcontrib = np.zeros(32, dtype=float)
    a_total = 0

    # Initialize a sub-dictionary to store answer_heatmaps per fact (by data_idx)
    model_answer_contribs = {}

    for idx, (relation_name, entries) in enumerate(relations_data.items()):
        
        if relation_name != "country_capital_city":
            continue

        # Group entries by data_idx
        data_idx_groups = {}
        for entry in entries:
            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()
            
            data_idx = entry['data_idx']
            if data_idx not in data_idx_groups:
                data_idx_groups[data_idx] = []
            data_idx_groups[data_idx].append(entry)

        # Process each data_idx group
        for data_idx, group_entries in data_idx_groups.items():
            # Determine the maximum shape
            ans_total = 0
            max_layers = max(entry['ffns_contribution_maps'].shape[0] for entry in group_entries)
            combined_map = np.zeros((max_layers, 32), dtype=float)  # Initialize with zeroes for contribution summing
            answer_contrib = np.zeros(32, dtype=float)  # Initialize answer_heatmap for the current group

            # Sum contributions across all entries in the group
            for g_entry in group_entries:
                current_map = g_entry['ffns_contribution_maps'].astype(float)
                # Pad current_map to max_heads if necessary
                if current_map.shape[0] < max_layers:
                    padded_map = np.zeros((max_layers, 32), dtype=float)
                    padded_map[:current_map.shape[0]] = current_map
                    current_map = padded_map
                combined_map += current_map  # Sum the contributions
            
            combined_map /= len(group_entries)    
            # Accumulate the combined map into the general heatmaps
            general_contrib += combined_map.sum(axis=0)
            g_total += combined_map.shape[0]

            for g_entry in group_entries:
                # Calculate and accumulate answer_heatmap based on answer_token_span of the current fact
                for t in g_entry["answer_token_span"]:
                    answer_contrib += combined_map[t - 1]
                answer_contrib /= len(g_entry["answer_token_span"])

                # Entity heatmap accumulation
                one_data_map = np.zeros(32, dtype=float)
                for t in g_entry["subj_token_span"]:
                    if t == 0:
                        e_total -= 1
                        continue
                    one_data_map += combined_map[t - 1]
                entity_contrib += one_data_map
                e_total += len(g_entry["subj_token_span"])

                # Accumulate to answer_allheatmap and entity heatmap
                one_data_map = np.zeros(32, dtype=float)
                for t in g_entry["answer_token_span"]:
                    one_data_map += combined_map[t - 1]
                entity_contrib += one_data_map
                e_total += len(g_entry["answer_token_span"])
                ans_total += len(g_entry["answer_token_span"])
                a_total += len(g_entry["answer_token_span"])

                for t in g_entry["answer_token_span"]:
                    answer_allcontrib += combined_map[t - 1]
            # Store the answer_heatmap for this specific fact (data_idx)
            answer_contrib /+ ans_total
            model_answer_contribs[data_idx] = answer_contrib    
                    
                        
    # Normalize heatmaps
    entity_contrib /= e_total
    answer_allcontrib /= a_total
    general_contrib /= g_total

    # Store heatmaps in the dictionary, including answer_heatmaps for each fact
    ffns_dict[model_name] = {
        'general_contrib': general_contrib,
        'entity_contrib': entity_contrib,
        'answer_contrib': model_answer_contribs,  # Store answer heatmaps for each fact
        'answer_allcontrib': answer_allcontrib
    }

In [ ]:
THRESHOLD = 0.2
fact_idx = 10

overlap_ffn = calculate_overlap_with_main_ffn(ffns_dict, THRESHOLD, fact_idx)
plot_proportion_overlap_multiple_ffn(overlap_ffn)
switches_results = calculate_switches_ffn(ffns_dict, THRESHOLD, fact_idx)
plot_proportion_switches_ffn(switches_results)

In [ ]:
all_contrib_values.shape

Conservative Calculation of FFN Contributions

1. AND over data_idx (different prompt, same fact)
2. AND over the token positions entites (subject and token pos), answers_all(answer token position of all facts),
3. AND over facts

In [ ]:
ffns_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]
    
    print(model_name)    
    g_total = 0
    general_contrib = np.zeros(32, dtype=bool)
    e_total = 0
    entity_contrib = np.zeros(32, dtype=bool)
    answer_allcontrib = np.zeros(32, dtype=bool)
    a_total = 0

    # Initialize a sub-dictionary to store answer_heatmaps per fact (by data_idx)
    model_answer_contribs = {}

    for idx, (relation_name, entries) in enumerate(relations_data.items()):
        
        if relation_name != "country_capital_city":
            continue

        # Group entries by data_idx
        data_idx_groups = {}
        for entry in entries:
            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()
            
            data_idx = entry['data_idx']
            if data_idx not in data_idx_groups:
                data_idx_groups[data_idx] = []
            data_idx_groups[data_idx].append(entry)
            
    #print(general_contrib)

    subject_maps = []
    entity_maps = []
    answer_all_maps = []

    # Process each data_idx group
    for data_idx, group_entries in data_idx_groups.items():
        answer_contrib = np.zeros(32, dtype=bool)
        answer_maps = []

        
        for g_entry in group_entries:
            answer_tokens = [x-1 for x in g_entry["answer_token_span"]]
            subj_tokens = [x - 1 for x in g_entry['subj_token_span']]
            answer_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][answer_tokens]))
            answer_all_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][answer_tokens]))
            subject_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][subj_tokens]))
            entity_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][subj_tokens+answer_tokens]))
    
    
        answer_contrib = reduce(np.logical_and, answer_maps)
        model_answer_contribs[data_idx] = answer_contrib

    reduced_maps = [reduce(np.logical_and, entry['ffns_contribution_maps']) for entry in group_entries]
    general_contrib = reduce(np.logical_and, reduced_maps)
    
    subject_contribs = reduce(np.logical_and, subject_maps)
    entity_contrib = reduce(np.logical_and, entity_maps)
    answer_allcontrib = reduce(np.logical_and, answer_all_maps)


    # Store heatmaps in the dictionary, including answer_heatmaps for each fact
    ffns_dict[model_name] = {
        'general_contrib': general_contrib,
        'entity_contrib': entity_contrib,
        'answer_contrib': model_answer_contribs,  # Store answer heatmaps for each fact
        'answer_allcontrib': answer_allcontrib
    }

In [ ]:
THRESHOLD

In [ ]:
overlap_ffn = calculate_overlap_with_main_ffn(ffns_dict, 0.9, fact_idx=0)
plot_proportion_overlap_multiple_ffn(overlap_ffn)

In [ ]:
switches_results = calculate_switches_ffn(ffns_dict, 0.1, fact_idx=0)
plot_proportion_switches_ffn(switches_results)

In [ ]:
for i in range(len(sorted_models) - 1):
        
        if sorted_models[i] == 'main':
            continue
        
        model_name = sorted_models[i]
        next_model_name = sorted_models[i + 1]
    
        relations_data = models_data[model_name]["country_capital_city"][0:25:5]
        next_model_relations_data = models_data[next_model_name]["country_capital_city"][0:25:5]
    
    # for model_name, model_data in heatmaps_dict.items():
    #    if model_name == 'main':
    #        continue

        # Store overlaps for this model
        for j in range(len(relations_data)):
            resid_top_tokens_cur = relations_data[j]["resid_top_tokens"]
            resid_top_tokens_next = next_model_relations_data[j]["resid_top_tokens"]
            
            ans_token = [i-1 for i in relations_data[j]['answer_token_span']]
            
            resid_top_tokens_cur[:, cur_ans_token] != resid_top_tokens_next[:, cur_ans_token]
            
            print("test")

        model_switches = {}
        
        for heatmap_type in model_data.keys():
            # Perform element-wise overlap calculation between model's heatmap and 'main' heatmap
            
            if heatmap_type == "answer_heatmap":
                switches_map = (next_model_data[heatmap_type][fact_idx] - model_data[heatmap_type][fact_idx]) > THRESHOLD #(model_data[heatmap_type][fact_idx] > THRESHOLD) != (next_model_data[heatmap_type][fact_idx] > THRESHOLD)
            else:
                switches_map = (next_model_data[heatmap_type] - model_data[heatmap_type]) > THRESHOLD #(model_data[heatmap_type] > THRESHOLD) != (next_model_data[heatmap_type] > THRESHOLD)
            
            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_switches = np.sum(switches_map) / switches_map.size
            
            # Store the overlap map and proportion
            model_switches[heatmap_type] = {
                'switches_map': switches_map,
                'proportion_switches': proportion_switches
            }
        
        # Add this model's overlap results to the main dictionary
        switches_results[model_name] = model_switches

this is the new version of the ffn_dict

In [ ]:
ffns_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]
    
    print(model_name)    
    g_total = 0
    general_contrib = np.zeros(32, dtype=float)
    e_total = 0
    entity_contrib = np.zeros(32, dtype=float)
    
    # Initialize dictionaries for specific contributions
    relation_answer_contribs = {}
    relation_answer_with_specific = {}

    for idx, (relation_name, entries) in enumerate(relations_data.items()):
        
        # Initialize contributions for this relation
        relation_answer_contrib = np.zeros(32, dtype=float)
        relation_answer_total = 0

        # Group entries by data_idx
        data_idx_groups = {}
        for entry in entries:
            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()
            
            data_idx = entry['data_idx']
            if data_idx not in data_idx_groups:
                data_idx_groups[data_idx] = []
            data_idx_groups[data_idx].append(entry)

        answer_specific_contribs = {}
        # Process each data_idx group
        for data_idx, group_entries in data_idx_groups.items():
            
            answer_specific_contrib = np.zeros(32, dtype=float)
            answer_specific_total = 0
            
            for g_entry in group_entries:
                current_map = g_entry['ffns_contribution_maps'].astype(float)

                # General contribution
                general_contrib += current_map.sum(axis=0)
                g_total += current_map.shape[0]

                # Entity contribution
                one_data_map = np.zeros(32, dtype=float)
                for t in g_entry["subj_token_span"]:
                    if t == 0:
                        e_total -= 1
                        continue
                    one_data_map += current_map[t - 1]
                entity_contrib += one_data_map
                e_total += len(g_entry["subj_token_span"])

                # Answer contribution
                one_data_map = np.zeros(32, dtype=float)
                for t in g_entry["answer_token_span"]:
                    one_data_map += current_map[t - 1]
                entity_contrib += one_data_map
                relation_answer_contrib += one_data_map
                answer_specific_contrib += one_data_map

                e_total += len(g_entry["answer_token_span"])
                relation_answer_total += len(g_entry["answer_token_span"])
                answer_specific_total += len(g_entry["answer_token_span"])
            
            # Normalize and store answer-specific contributions
            answer_specific_contrib /= answer_specific_total
            answer_specific_contribs[data_idx] = answer_specific_contrib
                
        # Normalize and store relation-specific contributions
        relation_answer_contrib /= relation_answer_total
        relation_answer_contribs[relation_name] = relation_answer_contrib
        relation_answer_with_specific[relation_name] = answer_specific_contribs

    # Normalize overall contributions
    general_contrib /= g_total
    entity_contrib /= e_total

    # Store contributions in the dictionary
    ffns_dict[model_name] = {
        'general_contrib': general_contrib,
        'entity_contrib': entity_contrib,
        'relation_answer_contribs': relation_answer_contribs,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_overlap_with_main_ffn(ffns_dict, THRESHOLD, relation_name, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's contributions
    main_contribs = ffns_dict['main']

    # Iterate through each model (excluding 'main') and each contribution type
    for model_name, model_data in ffns_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for contrib_type in main_contribs.keys():
            if contrib_type == "relation_answer_contribs":
                # Index by relation_name
                overlap_map = (model_data[contrib_type][relation_name] > THRESHOLD) == (main_contribs[contrib_type][relation_name] > THRESHOLD)
            elif contrib_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx
                overlap_map = (model_data[contrib_type][relation_name][fact_idx] > THRESHOLD) == (main_contribs[contrib_type][relation_name][fact_idx] > THRESHOLD)
            else:
                # General and entity contributions
                overlap_map = (model_data[contrib_type] > THRESHOLD) == (main_contribs[contrib_type] > THRESHOLD)
            
            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            
            # Store the overlap map and proportion
            model_overlaps[contrib_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

def calculate_switches_ffn(ffns_dict, THRESHOLD, relation_name, fact_idx):
    switches_results = {}
    
    # Iterate through sorted models in pairs
    for i in range(len(sorted_models) - 1):
        if sorted_models[i] == 'main':
            continue
        
        model_name = sorted_models[i]
        next_model_name = sorted_models[i + 1]
    
        model_data = ffns_dict[model_name]
        next_model_data = ffns_dict[next_model_name]

        # Store switches for this model
        model_switches = {}
        
        for contrib_type in model_data.keys():
            if contrib_type == "relation_answer_contribs":
                # Index by relation_name
                switches_map = (next_model_data[contrib_type][relation_name] - model_data[contrib_type][relation_name]) > THRESHOLD
            elif contrib_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx
                switches_map = (next_model_data[contrib_type][relation_name][fact_idx] - model_data[contrib_type][relation_name][fact_idx]) > THRESHOLD
            else:
                # General and entity contributions
                switches_map = (next_model_data[contrib_type] - model_data[contrib_type]) > THRESHOLD
            
            # Calculate summary statistics, e.g., proportion of switching positions
            proportion_switches = np.sum(switches_map) / switches_map.size
            
            # Store the switches map and proportion
            model_switches[contrib_type] = {
                'switches_map': switches_map,
                'proportion_switches': proportion_switches
            }
        
        # Add this model's switches results to the main dictionary
        switches_results[model_name] = model_switches
    
    return switches_results

In [ ]:
def plot_proportion_overlap_multiple_ffn(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    relation_answer_proportions = []
    relation_answer_specific_proportions = []

    # Populate lists for each contrib type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_contrib']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_contrib']['consistency'])
        relation_answer_proportions.append(overlap_results[model_name]['relation_answer_contribs'][relation_name]['consistency'])
        relation_answer_specific_proportions.append(overlap_results[model_name]['relation_answer_with_specific'][relation_name][fact_idx]['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Contribution", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Contribution", marker='s', linestyle='--')
    plt.plot(model_names, relation_answer_proportions, label=f"Relation Contribution ({relation_name})", marker='^', linestyle='-.')
    plt.plot(model_names, relation_answer_specific_proportions, label=f"Specific Contribution ({relation_name}, Fact {fact_idx})", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title(f"Consistency Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_proportion_switches_ffn(switches_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        switches_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    relation_answer_proportions = []
    relation_answer_specific_proportions = []

    # Populate lists for each contrib type
    for model_name in model_names:
        general_proportions.append(switches_results[model_name]['general_contrib']['proportion_switches'])
        entity_proportions.append(switches_results[model_name]['entity_contrib']['proportion_switches'])
        relation_answer_proportions.append(switches_results[model_name]['relation_answer_contribs'][relation_name]['proportion_switches'])
        relation_answer_specific_proportions.append(switches_results[model_name]['relation_answer_with_specific'][relation_name][fact_idx]['proportion_switches'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Contribution", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Contribution", marker='s', linestyle='--')
    plt.plot(model_names, relation_answer_proportions, label=f"Relation Contribution ({relation_name})", marker='^', linestyle='-.')
    plt.plot(model_names, relation_answer_specific_proportions, label=f"Specific Contribution ({relation_name}, Fact {fact_idx})", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Proportion Switches")
    plt.title(f"Proportion Switches Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
THRESHOLD = 0.2
relation_name = "country_capital_city"
fact_idx = 1
overlap = calculate_overlap_with_main_ffn(heatmaps_dict, THRESHOLD, relation_name, fact_idx)
switches = calculate_switches_ffn(heatmaps_dict, THRESHOLD, relation_name, fact_idx)


plot_proportion_overlap_multiple_ffn(overlap, relation_name, fact_idx)
plot_proportion_switches_ffn(switches, relation_name, fact_idx)

this is the new version of the more conservative calculation

In [ ]:
ffns_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]
    
    print(model_name)
    general_contrib = np.zeros(32, dtype=bool)
    entity_contrib = np.zeros(32, dtype=bool)

    # Initialize dictionaries for specific contributions
    relation_answer_contribs = {}
    relation_answer_with_specific = {}

    for idx, (relation_name, entries) in enumerate(relations_data.items()):
        
        # Initialize contributions for this relation
        relation_answer_contrib = np.zeros(32, dtype=bool)
        answer_all_maps = []
        subject_maps = []
        entity_maps = []

        # Group entries by data_idx
        data_idx_groups = {}
        for entry in entries:
            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()
            
            data_idx = entry['data_idx']
            if data_idx not in data_idx_groups:
                data_idx_groups[data_idx] = []
            data_idx_groups[data_idx].append(entry)

        answer_specific_contribs = {}
        # Process each data_idx group
        for data_idx, group_entries in data_idx_groups.items():
            answer_contrib = np.zeros(32, dtype=bool)
            answer_maps = []

            for g_entry in group_entries:
                answer_tokens = [x - 1 for x in g_entry["answer_token_span"]]
                subj_tokens = [x - 1 for x in g_entry["subj_token_span"]]

                # Answer maps
                answer_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][answer_tokens]))
                answer_all_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][answer_tokens]))
                
                # Subject maps
                subject_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][subj_tokens]))
                
                # Entity maps (subject + answer tokens)
                entity_maps.append(reduce(np.logical_and, g_entry['ffns_contribution_maps'][subj_tokens + answer_tokens]))
            
            # Calculate fact-specific answer contributions
            answer_contrib = reduce(np.logical_and, answer_maps)
            answer_specific_contribs[data_idx] = answer_contrib

        # Relation-specific contributions
        relation_answer_contrib = reduce(np.logical_and, answer_maps)
        relation_answer_contribs[relation_name] = relation_answer_contrib
        relation_answer_with_specific[relation_name] = answer_specific_contribs

    # General and entity-level contributions
    general_contrib = reduce(np.logical_and, [reduce(np.logical_and, entry['ffns_contribution_maps']) for entry in entries])
    subject_contribs = reduce(np.logical_and, subject_maps)
    entity_contrib = reduce(np.logical_and, entity_maps)
    answer_allcontrib = reduce(np.logical_and, answer_all_maps)

    # Store contributions in the dictionary
    ffns_dict[model_name] = {
        'general_contrib': general_contrib,
        'entity_contrib': entity_contrib,
        'relation_answer_contribs': relation_answer_contribs,
        'relation_answer_with_specific': relation_answer_with_specific,
        'answer_allcontrib': answer_allcontrib
    }

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Initialize a list to keep track of switches
layer_switches = {layer_idx: [] for layer_idx in range(65)}


for i in range(len(sorted_models) - 1):
    if sorted_models[i] == 'main':
        continue

    model_name = sorted_models[i]
    next_model_name = sorted_models[i + 1]

    # Get the relevant relations data for the current and next models
    relations_data = models_data[model_name]["country_capital_city"][2::4]
    next_model_relations_data = models_data[next_model_name]["country_capital_city"][2::4]

    # Loop through the relations data
    for j in range(len(relations_data)):
        # Extract the top tokens for the current and next model
        resid_top_tokens_cur = np.array(relations_data[j]["resid_top_tokens"])  # Assuming resid_top_tokens is a numpy array
        resid_top_tokens_next = np.array(next_model_relations_data[j]["resid_top_tokens"])

        # Get the answer token span for both models
        cur_ans_token = [i - 1 for i in relations_data[j]['answer_token_span']]

        # Loop through layers and print all tokens for each layer
        for layer_idx in range(resid_top_tokens_cur.shape[0]):  # Iterate through layers (first dimension: 65 layers)
            # Get tokens from the current and next model for the given layer
            cur_tokens_at_layer = [resid_top_tokens_cur[layer_idx, cur_ans_token][0][0]]#[:,0]
            next_tokens_at_layer = [resid_top_tokens_next[layer_idx, cur_ans_token][0][0]]#[:,0]

            # Convert all tokens to string using model.to_str_tokens() for both current and next model
            cur_tokens_str = [model.to_string([token]) for token in cur_tokens_at_layer]
            next_tokens_str = [model.to_string([token]) for token in next_tokens_at_layer]
            
            # Compare the tokens between the two models
            for cur_token, next_token in zip(cur_tokens_str, next_tokens_str):
                if cur_token != next_token:  # If the token switches
                    layer_switches[layer_idx].append((cur_token, next_token))  # Track the token pair (switch) for the current layer

            # Print the layer and all tokens
            print(f"Model: {model_name} vs {next_model_name}, Relation: {relations_data[j]['sentence']}, Layer: {layer_idx}")
            print(f"Current model tokens at layer {layer_idx}: {cur_tokens_str}")
            print(f"Next model tokens at layer {layer_idx}: {next_tokens_str}")
            print("-" * 50)

In [ ]:
# Plot histograms for each layer
for layer_idx, switches in layer_switches.items():
    if len(switches) > 0:  # Only plot if there are switches in this layer
        # Count the frequency of each switch pair in this layer
        switch_pair_counts = Counter(switches)

        # Filter the switches that have a count greater than 25
        filtered_switch_pairs = {pair: count for pair, count in switch_pair_counts.items() if count > 1}

        # If there are no pairs with count > 25, skip this layer
        if filtered_switch_pairs:
            # Plot the histogram of filtered switch pairs for this layer
            switch_labels = [f"{pair[0]} -> {pair[1]}" for pair in filtered_switch_pairs.keys()]
            switch_values = list(filtered_switch_pairs.values())

            # Plotting the switch pairs for this layer
            plt.figure(figsize=(12, 6))
            plt.barh(switch_labels, switch_values)
            plt.xlabel('Frequency of Switch')
            plt.ylabel('Token Pair (Switches)')
            plt.title(f'Histogram of Token Switches (Count > 25) for Layer {layer_idx}')
            plt.show()

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# Initialize a dictionary to track token pairs (switches) per layer
layer_switches = {'resid_mid': [], 'resid_post': [], 'final_residual_stream': []}

for i in range(len(sorted_models) - 1):
    if sorted_models[i] == 'main':
        continue

    model_name = sorted_models[i]
    next_model_name = sorted_models[i + 1]

    # Get the relevant relations data for the current and next models
    relations_data = models_data[model_name]["country_capital_city"][1::4]
    next_model_relations_data = models_data[next_model_name]["country_capital_city"][1::4]

    # Loop through the relations data
    for j in range(len(relations_data)):
        # Extract the top tokens for the current and next model
        resid_top_tokens_cur = np.array(relations_data[j]["resid_top_tokens"])  # Assuming resid_top_tokens is a numpy array
        resid_top_tokens_next = np.array(next_model_relations_data[j]["resid_top_tokens"])

        # Get the answer token span for both models
        cur_ans_token = [i - 1 for i in relations_data[j]['answer_token_span']]

        # Loop through layers and print all tokens for each layer
        for layer_idx in range(resid_top_tokens_cur.shape[0]):  # Iterate through layers (first dimension: 65 layers)
            # Get tokens from the current and next model for the given layer
            cur_tokens_at_layer = [resid_top_tokens_cur[layer_idx, cur_ans_token][0][0]]#[:,0]
            next_tokens_at_layer = [resid_top_tokens_next[layer_idx, cur_ans_token][0][0]]#[:,0]

            # Convert all tokens to string using model.to_str_tokens() for both current and next model
            cur_tokens_str = [model.to_string([token]) for token in cur_tokens_at_layer]
            next_tokens_str = [model.to_string([token]) for token in next_tokens_at_layer]
            
            # Compare the tokens between the two models and track switches per layer
            for cur_token, next_token in zip(cur_tokens_str, next_tokens_str):
                if cur_token != next_token:  # If the token switches
                    # Track the token pair (switch) for the appropriate layer
                    if layer_idx == 0:  # If it's the resid_mid layer
                        layer_switches['resid_mid'].append((cur_token, next_token))
                    elif layer_idx == 1:  # If it's the resid_post layer
                        layer_switches['resid_post'].append((cur_token, next_token))
                    elif layer_idx == resid_top_tokens_cur.shape[0] - 1:  # If it's the final residual stream
                        layer_switches['final_residual_stream'].append((cur_token, next_token))


            # Print the layer and all tokens
            print(f"Model: {model_name} vs {next_model_name}, Relation: {relations_data[j]['sentence']}, Layer: {layer_idx}")
            print(f"Current model tokens at layer {layer_idx}: {cur_tokens_str}")
            print(f"Next model tokens at layer {layer_idx}: {next_tokens_str}")
            print("-" * 50)

In [ ]:
for layer_name, switches in layer_switches.items():
    if len(switches) > 0:  # Only plot if there are switches in this layer
        # Count the frequency of each switch pair in this layer
        switch_pair_counts = Counter(switches)

        # Filter the switches that have a count greater than 25
        filtered_switch_pairs = {pair: count for pair, count in switch_pair_counts.items() if count > 0}

        # If there are no pairs with count > 25, skip this layer
        if filtered_switch_pairs:
            # Plot the histogram of filtered switch pairs for this layer
            switch_labels = [f"{pair[0]} -> {pair[1]}" for pair in filtered_switch_pairs.keys()]
            switch_values = list(filtered_switch_pairs.values())

            # Plotting the switch pairs for this layer
            plt.figure(figsize=(12, 50))
            plt.barh(switch_labels, switch_values)
            plt.xlabel('Frequency of Switch')
            plt.ylabel('Token Pair (Switches)')
            plt.title(f'Histogram of Token Switches (Count > 25) for {layer_name}')
            plt.show()

In [ ]:
model.to_string(5009)